In [ ]:
import numpy as np

def RotM(axis,angle):
    """
    Rotation matrices in standard convention.
    Args:
        axis : specify rotation axis string in 'x', 'y', 'z'.
        angle (float): Rotation angle.

    Returns:
        3x3 np-array: rotation matrix.
    """
    if (axis=='x'):
        return np.asarray([[1, 0            ,  0            ],
                           [0, np.cos(angle), -np.sin(angle)],
                           [0, np.sin(angle),  np.cos(angle)]])
    elif (axis=='y'):
        return np.asarray([[ np.cos(angle), 0,  np.sin(angle) ],
                           [ 0            , 1,  0             ],
                           [-np.sin(angle), 0,  np.cos(angle) ]])
    elif (axis=='z'):
        return np.asarray([[ np.cos(angle), -np.sin(angle),  0 ],
                           [ np.sin(angle),  np.cos(angle),  0 ],
                           [ 0            ,  0            ,  1 ]]) 

In [ ]:
### Transformation function
def transform_vector_field(R,offset,vector,selection=(0,1,2)):
    """
    Tranform a vector field V=`vector` according to the general rotation V'(x')=R*V(RT*x'+offset),
    coordinate notation follows the documents.
    R - rotation matrix
    offset - offset vector
    vector - vector field represented by a function with 3 components taking (x,t) arguments, x is 3-component spatial, t is time
    selection - select output projections, labeled by components 0, 1, 2, allows permutations and dimension reduction

    output - vector field represented by a function inlcuding selection
    """
    # def vector_transformed(x_):
    #     return np.asarray([(R@vector((np.transpose(R)@x_) + offset))[idx] for idx in selection])
    def vector_field_transformed(x_,t_):
        return np.asarray([(np.transpose(R) @ vector(R@(x_ - offset),t_))[idx] for idx in selection])
    return vector_field_transformed
    

In [ ]:
# Rotation matrices
def RotM(axis,angle):
    """
    Rotation matrices in standard convention.
    Args:
        axis : specify rotation axis string in 'x', 'y', 'z'.
        angle (float): Rotation angle.

    Returns:
        3x3 np-array: rotation matrix.
    """
    if (axis=='x'):
        return np.asarray([[1, 0            ,  0            ],
                           [0, np.cos(angle), -np.sin(angle)],
                           [0, np.sin(angle),  np.cos(angle)]])
    elif (axis=='y'):
        return np.asarray([[ np.cos(angle), 0,  np.sin(angle) ],
                           [ 0            , 1,  0             ],
                           [-np.sin(angle), 0,  np.cos(angle) ]])
    elif (axis=='z'):
        return np.asarray([[ np.cos(angle), -np.sin(angle),  0 ],
                           [ np.sin(angle),  np.cos(angle),  0 ],
                           [ 0            ,  0            ,  1 ]]) 


Full tranform according to Smilei

In [ ]:
l0 = 2.0*np.pi              # laser wavelength
t0 = l0                       # optical cicle
Lsim = [10.*l0,8.*l0,5.*l0]  # length of the simulation
Tsim = 18.*t0                 # duration of the simulation
resx = 12.                    # nb of cells in one laser wavelength
rest = 22.                    # nb of timesteps in one optical cycle 

ang = [-np.pi/7., np.pi/6.] # angle as defined by the smilei tutorial
focus = [0.5*Lsim[0], 0.5*Lsim[1], 0.5*Lsim[2]] # the position of the focus in the Smilei coordinates = offset in the 

## General Gaussian beam represented by B-field
waist = l0
a0    = 1.
omega = 1.

fwhm = 6.*t0
def time_envelope(t):
    sigma = (0.5*fwhm)**2/np.log(2.0)
    return np.exp( -(t)**2 / sigma )

# derived quatities
k0 = 1.
zR = omega * waist**2/2.

def B_Gauss_lin(x, t):

    r2 = x[1]**2 + x[2]**2
    w = l0 * np.sqrt(1.0 + (x[0]/zR)**2)
    invR = x[0] / (x[0]**2 + zR**2)
    gouy = np.arctan(x[0] / zR)
    
    phase = (omega*t - k0*x[0] - 0.5 * r2 * invR  + gouy)
    envelope = ((waist/w)*np.exp(-r2/w**2)*time_envelope(t - x[0])
    )

    return a0 * envelope * np.cos(phase)
    
def Bfield(x,t):
    return np.asarray([
        0.,
        B_Gauss_lin(x, t),
        0.        
        ])

xfix = 0.
def Bfield_xplane(xfix):
    def B1(y_,z_,t_):
        return transform_vector_field(RotM('z',ang[1])@RotM('y',ang[0]),focus,Bfield,selection=(1,))(np.asarray([xfix,y_,z_]),t_)
    def B2(y_,z_,t_):
        return transform_vector_field(RotM('z',ang[1])@RotM('y',ang[0]),focus,Bfield,selection=(2,))(np.asarray([xfix,y_,z_]),t_)
    return [B1, B2]

In [ ]:
field_plane = Bfield_xplane(0.)
print(field_plane[0](0.,0.,0.))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

N = 250

x0_vals = np.linspace(-Lsim[0], Lsim[0], N)
x1_vals = np.linspace(-Lsim[1], Lsim[1], N)

X0, X1 = np.meshgrid(x0_vals, x1_vals)

def plot_field(t, x2):
    field = B_Gauss_lin([X0, X1, x2], t)

    plt.figure(figsize=(8, 6))
    im = plt.imshow(
        field,
        extent=[-Lsim[0], Lsim[0], -Lsim[1], Lsim[1]],
        origin='lower',
        aspect='auto'
    )

    plt.xlabel('x[0]')
    plt.ylabel('x[1]')
    plt.title(f'B_Gauss_lin, t={t:.3f}, x[2]={x2:.3f}')
    plt.colorbar(im, label='B')
    plt.tight_layout()
    plt.show()

interact(
    plot_field,
    t=FloatSlider(
        min=-Tsim,
        max=Tsim,
        step=Tsim/200,
        value=0.0,
        description='t'
    ),
    x2=FloatSlider(
        min=-Lsim[2],
        max=Lsim[2],
        step=(2*Lsim[2])/200,
        value=0.0,
        description='x[2]'
    )
);

Test rotations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

N = 250

x0_vals = np.linspace(-Lsim[0], Lsim[0], N)
x1_vals = np.linspace(-Lsim[1], Lsim[1], N)

X0, X1 = np.meshgrid(x0_vals, x1_vals)

def Bfield_meshed(x, t):

    By = B_Gauss_lin(x, t)

    return np.asarray([
        np.zeros_like(By),
        By,
        np.zeros_like(By)
    ])
    
def transform_vector_field_vectorised(R,offset,vector,selection=(0,1,2)):
    """
    The same as transform_vector_field but using shape castings.
    """
    def vector_field_transformed(x_,t_):
        x_rot = np.einsum('ij,j...->i...', R, x_ - offset)
        vec = vector(x_rot, t_)
        vec_rot = np.einsum('ij,j...->i...', R.T, vec)
        return np.asarray([vec_rot[idx] for idx in selection])
    return vector_field_transformed
    

def rotated_field(x0,x1,x2,t,theta,phi):
    return transform_vector_field_vectorised(RotM('z',theta)@RotM('y',phi),0.,Bfield_meshed,selection=(1,))(np.asarray([x0,x1,np.full_like(x0,x2)]),t)

def plot_field(t, x2, theta, phi):
    # field = B_Gauss_lin([X0, X1, x2], t)
    # field = transform_vector_field(RotM('z',theta)@RotM('y',phi),focus,Bfield,selection=(1,))(np.asarray([X0,X1,x2]),t_)
    field =  np.squeeze(rotated_field(X0,X1,x2,t,theta,phi))

    plt.figure(figsize=(8, 6))
    im = plt.imshow(
        field,
        extent=[-Lsim[0], Lsim[0], -Lsim[1], Lsim[1]],
        origin='lower',
        aspect='auto'
    )

    plt.xlabel('x[0]')
    plt.ylabel('x[1]')
    plt.title(f'B_Gauss_lin, t={t:.3f}, x[2]={x2:.3f}')
    plt.colorbar(im, label='B')
    plt.tight_layout()
    plt.show()

interact(
    plot_field,
    t=FloatSlider(
        min=-Tsim,
        max=Tsim,
        step=Tsim/200,
        value=0.0,
        description='t'
    ),
    x2=FloatSlider(
        min=-Lsim[2],
        max=Lsim[2],
        step=(2*Lsim[2])/200,
        value=0.0,
        description='x[2]'
    ),
    theta=FloatSlider(
        min=-np.pi,
        max=np.pi,
        step=(2*np.pi)/200,
        value=0.0,
        description='theta'
    ),
    phi=FloatSlider(
        min=-np.pi,
        max=np.pi,
        step=(2*np.pi)/200,
        value=0.0,
        description='phi'
    )
);

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

N = 250

x0_vals = np.linspace(-Lsim[0], Lsim[0], N)
x1_vals = np.linspace(-Lsim[1], Lsim[1], N)

X0, X1 = np.meshgrid(x0_vals, x1_vals)

def Bfield_meshed(x, t):

    By = B_Gauss_lin(x, t)

    return np.asarray([
        np.zeros_like(By),
        By,
        np.zeros_like(By)
    ])
    
def transform_vector_field_vectorised(R,offset,vector,selection=(0,1,2)):
    """
    The same as transform_vector_field but using shape castings.
    """
    def vector_field_transformed(x_,t_):
        off = np.asarray(offset).reshape((3,) + (1,)*(x_.ndim-1))
        x_rot = np.einsum('ij,j...->i...', R, x_ - off)
        vec = vector(x_rot, t_)
        vec_rot = np.einsum('ij,j...->i...', R.T, vec)
        return np.asarray([vec_rot[idx] for idx in selection])
    return vector_field_transformed
    

def rotated_field_offset(x0,x1,x2,t,theta,phi,off0,off1,off2):
    return transform_vector_field_vectorised(RotM('z',theta)@RotM('y',phi),np.asarray([off0,off1,off2]),Bfield_meshed,selection=(1,))(np.asarray([x0,x1,np.full_like(x0,x2)]),t)

def plot_field(t, x2, theta, phi, off0, off1, off2):
    # field = B_Gauss_lin([X0, X1, x2], t)
    # field = transform_vector_field(RotM('z',theta)@RotM('y',phi),focus,Bfield,selection=(1,))(np.asarray([X0,X1,x2]),t_)
    field =  np.squeeze(rotated_field_offset(X0,X1,x2,t,theta,phi,off0,off1,off2))

    plt.figure(figsize=(8, 6))
    im = plt.imshow(
        field,
        extent=[-Lsim[0], Lsim[0], -Lsim[1], Lsim[1]],
        origin='lower',
        aspect='auto'
    )

    plt.xlabel('x[0]')
    plt.ylabel('x[1]')
    plt.title(f'B_Gauss_lin, t={t:.3f}, x[2]={x2:.3f}')
    plt.colorbar(im, label='B')
    plt.tight_layout()
    plt.show()

interact(
    plot_field,
    t=FloatSlider(
        min=-Tsim,
        max=Tsim,
        step=Tsim/200,
        value=0.0,
        description='t'
    ),
    x2=FloatSlider(
        min=-Lsim[2],
        max=Lsim[2],
        step=(2*Lsim[2])/200,
        value=0.0,
        description='x[2]'
    ),
    theta=FloatSlider(
        min=-np.pi,
        max=np.pi,
        step=(2*np.pi)/200,
        value=0.0,
        description='theta'
    ),
    phi=FloatSlider(
        min=-np.pi,
        max=np.pi,
        step=(2*np.pi)/200,
        value=0.0,
        description='phi'
    ),
    off0=FloatSlider(
        min=-Lsim[0],
        max=Lsim[0],
        step=(2*Lsim[0])/200,
        value=0.0,
        description='off0'
    ),
    off1=FloatSlider(
        min=-Lsim[1],
        max=Lsim[1],
        step=(2*Lsim[1])/200,
        value=0.0,
        description='off1'
    ),
    off2=FloatSlider(
        min=-Lsim[2],
        max=Lsim[2],
        step=(2*Lsim[2])/200,
        value=0.0,
        description='off2'
    ),
);

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

N = 250

x0_vals = np.linspace(-Lsim[0], Lsim[0], N)
x1_vals = np.linspace(-Lsim[1], Lsim[1], N)
x2_vals = np.linspace(-Lsim[2], Lsim[2], N)

# x0-x1 plane
X01_0, X01_1 = np.meshgrid(x0_vals, x1_vals)

# x0-x2 plane
X02_0, X02_2 = np.meshgrid(x0_vals, x2_vals)

# x1-x2 plane
X12_1, X12_2 = np.meshgrid(x1_vals, x2_vals)


def Bfield_meshed(x, t):

    By = B_Gauss_lin(x, t)

    return np.asarray([
        np.zeros_like(By),
        By,
        np.zeros_like(By)
    ])


def transform_vector_field_vectorised(
    R,
    offset,
    vector,
    selection=(0, 1, 2)
):
    """
    The same as transform_vector_field but using shape castings.
    """

    def vector_field_transformed(x_, t_):

        off = np.asarray(offset).reshape(
            (3,) + (1,) * (x_.ndim - 1)
        )

        x_rot = np.einsum(
            'ij,j...->i...',
            R,
            x_ - off
        )

        vec = vector(x_rot, t_)

        vec_rot = np.einsum(
            'ij,j...->i...',
            R.T,
            vec
        )

        return np.asarray(
            [vec_rot[idx] for idx in selection]
        )

    return vector_field_transformed


def rotated_field_offset(
    x0,
    x1,
    x2,
    t,
    theta,
    phi,
    off0,
    off1,
    off2
):
    return transform_vector_field_vectorised(
        RotM('z', theta) @ RotM('y', phi),
        np.asarray([off0, off1, off2]),
        Bfield_meshed,
        selection=(1,)
    )(
        np.asarray([
            x0,
            x1,
            np.full_like(x0, x2)
        ]),
        t
    )


def plot_field_3planes(
    t,
    x0,
    x1,
    x2,
    theta,
    phi,
    off0,
    off1,
    off2
):

    # ---------------------------------------------------------
    # x0-x1 plane (x2 fixed)
    # ---------------------------------------------------------

    field01 = np.squeeze(
        rotated_field_offset(
            X01_0,
            X01_1,
            x2,
            t,
            theta,
            phi,
            off0,
            off1,
            off2
        )
    )

    # ---------------------------------------------------------
    # x0-x2 plane (x1 fixed)
    # ---------------------------------------------------------

    field02 = np.squeeze(
        transform_vector_field_vectorised(
            RotM('z', theta) @ RotM('y', phi),
            np.asarray([off0, off1, off2]),
            Bfield_meshed,
            selection=(1,)
        )(
            np.asarray([
                X02_0,
                np.full_like(X02_0, x1),
                X02_2
            ]),
            t
        )
    )

    # ---------------------------------------------------------
    # x1-x2 plane (x0 fixed)
    # ---------------------------------------------------------

    field12 = np.squeeze(
        transform_vector_field_vectorised(
            RotM('z', theta) @ RotM('y', phi),
            np.asarray([off0, off1, off2]),
            Bfield_meshed,
            selection=(1,)
        )(
            np.asarray([
                np.full_like(X12_1, x0),
                X12_1,
                X12_2
            ]),
            t
        )
    )

    # ---------------------------------------------------------
    # Dynamic symmetric colour scales
    # ---------------------------------------------------------

    lim01 = np.max(np.abs(field01))
    lim02 = np.max(np.abs(field02))
    lim12 = np.max(np.abs(field12))

    fig, axs = plt.subplots(
        1,
        3,
        figsize=(18, 6),
        constrained_layout=True
    )

    # ---------------------------------------------------------
    # x0-x1
    # ---------------------------------------------------------

    im0 = axs[0].imshow(
        field01,
        extent=[
            -Lsim[0], Lsim[0],
            -Lsim[1], Lsim[1]
        ],
        origin='lower',
        aspect='auto',
        vmin=-lim01,
        vmax=lim01
    )

    axs[0].set_title(
        f'x0-x1 plane, x2={x2:.3f}'
    )
    axs[0].set_xlabel('x0')
    axs[0].set_ylabel('x1')

    cbar0 = fig.colorbar(
        im0,
        ax=axs[0],
        orientation='horizontal',
        pad=0.12
    )
    cbar0.set_label('B')

    # ---------------------------------------------------------
    # x0-x2
    # ---------------------------------------------------------

    im1 = axs[1].imshow(
        field02,
        extent=[
            -Lsim[0], Lsim[0],
            -Lsim[2], Lsim[2]
        ],
        origin='lower',
        aspect='auto',
        vmin=-lim02,
        vmax=lim02
    )

    axs[1].set_title(
        f'x0-x2 plane, x1={x1:.3f}'
    )
    axs[1].set_xlabel('x0')
    axs[1].set_ylabel('x2')

    cbar1 = fig.colorbar(
        im1,
        ax=axs[1],
        orientation='horizontal',
        pad=0.12
    )
    cbar1.set_label('B')

    # ---------------------------------------------------------
    # x1-x2
    # ---------------------------------------------------------

    im2 = axs[2].imshow(
        field12,
        extent=[
            -Lsim[1], Lsim[1],
            -Lsim[2], Lsim[2]
        ],
        origin='lower',
        aspect='auto',
        vmin=-lim12,
        vmax=lim12
    )

    axs[2].set_title(
        f'x1-x2 plane, x0={x0:.3f}'
    )
    axs[2].set_xlabel('x1')
    axs[2].set_ylabel('x2')

    cbar2 = fig.colorbar(
        im2,
        ax=axs[2],
        orientation='horizontal',
        pad=0.12
    )
    cbar2.set_label('B')

    plt.show()


interact(
    plot_field_3planes,

    t=FloatSlider(
        min=-Tsim,
        max=Tsim,
        step=Tsim / 200,
        value=0.0,
        description='t'
    ),

    x0=FloatSlider(
        min=-Lsim[0],
        max=Lsim[0],
        step=(2 * Lsim[0]) / 200,
        value=0.0,
        description='x0'
    ),

    x1=FloatSlider(
        min=-Lsim[1],
        max=Lsim[1],
        step=(2 * Lsim[1]) / 200,
        value=0.0,
        description='x1'
    ),

    x2=FloatSlider(
        min=-Lsim[2],
        max=Lsim[2],
        step=(2 * Lsim[2]) / 200,
        value=0.0,
        description='x2'
    ),

    theta=FloatSlider(
        min=-np.pi,
        max=np.pi,
        step=(2 * np.pi) / 200,
        value=0.0,
        description='theta'
    ),

    phi=FloatSlider(
        min=-np.pi,
        max=np.pi,
        step=(2 * np.pi) / 200,
        value=0.0,
        description='phi'
    ),

    off0=FloatSlider(
        min=-Lsim[0],
        max=Lsim[0],
        step=(2 * Lsim[0]) / 200,
        value=0.0,
        description='off0'
    ),

    off1=FloatSlider(
        min=-Lsim[1],
        max=Lsim[1],
        step=(2 * Lsim[1]) / 200,
        value=0.0,
        description='off1'
    ),

    off2=FloatSlider(
        min=-Lsim[2],
        max=Lsim[2],
        step=(2 * Lsim[2]) / 200,
        value=0.0,
        description='off2'
    )
);